In [9]:
import pandas as pd
import numpy as np

In [10]:
df_raw = pd.read_csv('../../dataset/original/emails.csv')
print(len(df_raw.columns))
print(df_raw.info())
print(df_raw.head())

27
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123389 entries, 0 to 123388
Data columns (total 27 columns):
 #   Column                                Non-Null Count   Dtype 
---  ------                                --------------   ----- 
 0   Co_Ref                                123389 non-null  object
 1   Time_to_Renewal                       123389 non-null  object
 2   crm_accreditation_completed           102354 non-null  object
 3   crm_timely_completion                 102354 non-null  object
 4   crm_progress_towards_accreditation    102354 non-null  object
 5   crm_delays_in_accreditation           102354 non-null  object
 6   crm_contractor_suggested_leave        102354 non-null  object
 7   crm_contractor_engagement             102354 non-null  object
 8   crm_contractor_sentiment              102354 non-null  object
 9   crm_contractor_sentiment_score        102354 non-null  object
 10  crm_dts_or_ssip_mentioned             102354 non-null  object
 11  crm_custom

C:\Users\Asus\AppData\Local\Temp\ipykernel_34220\4105296966.py:1: DtypeWarning: Columns (16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../../dataset/original/emails.csv')


In [11]:
df_cols = df_raw.columns
print(df_cols)
# //save as a csv file
df_cols_df = pd.DataFrame(df_cols, columns=['Columns'])
df_cols_df.to_csv('../../dataset/01_raw/emails/columns.csv', index=True)

Index(['Co_Ref', 'Time_to_Renewal', 'crm_accreditation_completed',
       'crm_timely_completion', 'crm_progress_towards_accreditation',
       'crm_delays_in_accreditation', 'crm_contractor_suggested_leave',
       'crm_contractor_engagement', 'crm_contractor_sentiment',
       'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned',
       'crm_customer_payment_intention', 'crm_competitors_mentioned',
       'crm_membership_level', 'crm_platform_issues_raised',
       'crm_agent_chased_contractor', 'crm_agent_chase_count',
       'crm_accreditation_issues', 'crm_membership_overdue',
       'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price',
       'crm_customer_complained', 'crm_refund_mentioned',
       'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
       'crm_financial_hardship_mentioned', 'year'],
      dtype='object')


## Analysing the Null Cols

In [12]:
print(df_raw.isnull().sum())

Co_Ref                                      0
Time_to_Renewal                             0
crm_accreditation_completed             21035
crm_timely_completion                   21035
crm_progress_towards_accreditation      21035
crm_delays_in_accreditation             21035
crm_contractor_suggested_leave          21035
crm_contractor_engagement               21035
crm_contractor_sentiment                21035
crm_contractor_sentiment_score          21035
crm_dts_or_ssip_mentioned               21035
crm_customer_payment_intention          21035
crm_competitors_mentioned               11155
crm_membership_level                    11155
crm_platform_issues_raised              11155
crm_agent_chased_contractor             11155
crm_agent_chase_count                   11155
crm_accreditation_issues                11155
crm_membership_overdue                  11155
crm_auto_renewal_status                 11155
crm_dissatisified_with_renewal_price    11155
crm_customer_complained           

In [13]:
null_columns = df_raw.columns[df_raw.isnull().any()]
print("Columns with null values:", null_columns)

#save the null cols as a csv file
null_cols_df = pd.DataFrame(null_columns, columns=['Null Columns']) 
null_cols_df.to_csv('../../dataset/01_raw/emails/null_columns.csv', index=True)


Columns with null values: Index(['crm_accreditation_completed', 'crm_timely_completion',
       'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
       'crm_contractor_suggested_leave', 'crm_contractor_engagement',
       'crm_contractor_sentiment', 'crm_contractor_sentiment_score',
       'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
       'crm_competitors_mentioned', 'crm_membership_level',
       'crm_platform_issues_raised', 'crm_agent_chased_contractor',
       'crm_agent_chase_count', 'crm_accreditation_issues',
       'crm_membership_overdue', 'crm_auto_renewal_status',
       'crm_dissatisified_with_renewal_price', 'crm_customer_complained',
       'crm_refund_mentioned', 'crm_negative_customer_experience',
       'crm_dissatisfaction_with_support', 'crm_financial_hardship_mentioned'],
      dtype='object')


## Getting the datatype of each columns and uniques values if (more than 10 values then as ranges)

In [14]:



def is_date_column(series):
    return pd.api.types.is_datetime64_any_dtype(series)


def is_coref_column(series, threshold=50):
    num_unique = series.nunique(dropna=True)
    return series.dtype == 'object' and num_unique > threshold


def classify_column(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if series.dtype == 'object':
        # Try parsing to datetime
        try:
            parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
            if all(parsed.dt.time == pd.Timestamp("00:00:00").time()):
                return "date"
            else:
                return "datetime"
        except:
            pass

        # Try parsing to time
        try:
            pd.to_datetime(series.dropna().head(10), format='%H:%M:%S', errors='raise')
            return "time"
        except:
            pass

        return "categorical"

    return "categorical"


def get_unique_representation(series):
    unique_vals = series.dropna().unique()
    num_unique = len(unique_vals)

    if is_date_column(series) or is_coref_column(series):
        return []

    if pd.api.types.is_numeric_dtype(series):
        if num_unique <= 10:
            return sorted(unique_vals.tolist())
        else:
            return f"{series.min()} - {series.max()}"

    return unique_vals.tolist()


def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "unique_values": get_unique_representation(series),
            "null_count": series.isna().sum(),
            "num_unique": series.nunique(dropna=True)
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)


def save_validation_report(df, output_path):
    dt_val_df = create_data_validation_df(df)
    print(dt_val_df)

    dt_val_df.to_csv(output_path, index=True)
    print(f"\nReport saved to: {output_path}")


# output_file = './dataset/01_raw/ad_hoc/data_validation_summary.csv'
# save_validation_report(df_raw, output_file)

In [15]:
def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "null_count": series.isna().sum(),
            "null_percentage": round((series.isna().sum() / len(df)) * 100, 2),
            "num_unique": series.nunique(dropna=True),
            "unique_values": get_unique_representation(series),
            "top_values": series.value_counts(dropna=True).head(5).to_dict(),
            "skewness": series.skew() if pd.api.types.is_numeric_dtype(series) else None,
            "constant_column": series.nunique(dropna=True) <= 1
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)

df_validation_df = create_data_validation_df(df_raw)
print(df_validation_df.head())
df_validation_df.to_csv('../../dataset/01_raw/emails/data_validation_summary.csv', index=True)




C:\Users\Asus\AppData\Local\Temp\ipykernel_34220\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_34220\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_34220\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_

                          column_name data_type  column_type  null_count  \
0                              Co_Ref    object  categorical           0   
1                     Time_to_Renewal    object  categorical           0   
2         crm_accreditation_completed    object  categorical       21035   
3               crm_timely_completion    object  categorical       21035   
4  crm_progress_towards_accreditation    object  categorical       21035   

   null_percentage  num_unique                              unique_values  \
0             0.00       37964                                         []   
1             0.00           4  [pre_renewal, 14_out, prior_year, 45_out]   
2            17.05           3                   [Not Discussed, No, Yes]   
3            17.05           3                   [Not Discussed, No, Yes]   
4            17.05           3                   [Not Discussed, Yes, No]   

                                          top_values  skewness  \
0  {'AV6630': 

C:\Users\Asus\AppData\Local\Temp\ipykernel_34220\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_34220\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')


In [16]:
def check_mixed_data_types(df):
    mixed_summary = []

    for col in df.columns:
        series = df[col].dropna()

        # Try numeric conversion
        converted = pd.to_numeric(series, errors='coerce')

        # Count valid numeric vs invalid
        numeric_count = converted.notna().sum()
        non_numeric_count = converted.isna().sum()

        # If both exist → mixed
        if numeric_count > 0 and non_numeric_count > 0:
            sample_numeric = series[converted.notna()].iloc[0]
            sample_non_numeric = series[converted.isna()].iloc[0]

            mixed_summary.append({
                "column": col,
                "numeric_count": int(numeric_count),
                "non_numeric_count": int(non_numeric_count),
                "sample_numeric": sample_numeric,
                "sample_non_numeric": sample_non_numeric
            })

    return pd.DataFrame(mixed_summary)

mixed_df = check_mixed_data_types(df_raw)
print(mixed_df)

mixed_df.to_csv('../../dataset/01_raw/emails/mixed_data_types.csv', index=False)

                                 column  numeric_count  non_numeric_count  \
0             crm_contractor_engagement              1             102353   
1        crm_contractor_sentiment_score          71159              31195   
2             crm_dts_or_ssip_mentioned             20             102334   
3        crm_customer_payment_intention              4             102350   
4                 crm_agent_chase_count         111696                538   
5                crm_membership_overdue              1             112233   
6               crm_auto_renewal_status         112185                 49   
7  crm_dissatisified_with_renewal_price             40             112194   

  sample_numeric                                sample_non_numeric  
0              0                                               Yes  
1             50                                     Not Discussed  
2             20                                                No  
3             20              

## =================================================================================